# Foundry IQ Cookbook — Build a Grounded Knowledge Base

Modern agents need more than vector search — they need retrieval that **reasons over the question**, runs **parallel subqueries**, **reranks** with a semantic ranker, and **synthesizes** an answer with citations.

**Foundry IQ** is Microsoft's intelligence layer for agentic retrieval, built on Azure AI Search. This cookbook walks you through the full pipeline end-to-end using the Azure AI Search Python SDK (`12.1.0a*` alpha — the `2025-11-01-preview` API surface).

By the end of this notebook you will have:

1. Provisioned a **Search Index** with vector + semantic configuration
2. Indexed the NASA *Earth at Night* dataset (no separate embedding pipeline — the indexer vectorizes for you)
3. Created a **Knowledge Source** pointing at the index
4. Created a **Knowledge Base** that pairs the source with a chat model and answer synthesis
5. Run a **complex multi-part query** and inspected the planner's subqueries and citations
6. Continued the conversation with a **multi-turn follow-up** that preserves context
7. **Cleaned up** every resource the notebook created

## Architecture

```text
            ┌──────────────────────────────────────────────────────┐
            │                  Foundry IQ Pipeline                 │
            └──────────────────────────────────────────────────────┘
   query →  Knowledge Base ──► LLM planner ──► parallel subqueries ─┐
                │                                                   ▼
                │                                          Knowledge Source
                │                                                   │
                ▼                                                   ▼
        Answer synthesis ◄── reranked results ◄── Search Index (vector + semantic)
                │
                ▼
        Cited answer + activity trace
```


## Prerequisites

- An **Azure AI Search** service in a [region that supports agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support).
- A **Microsoft Foundry** (or Azure OpenAI) resource with two deployments:
  - An embedding model — `text-embedding-3-large` recommended (3072 dimensions, used in this notebook).
  - A chat completion model — `gpt-4o`, `gpt-4o-mini`, or `gpt-5-mini` all work.
- Admin keys for both services (this notebook uses key auth for portability; production deployments should prefer managed identity — see the [Foundry IQ private setup guide](https://learn.microsoft.com/azure/search/search-security-rbac)).

### Configure your environment

Copy `.env.example` to `.env` in this folder and fill in your endpoints and keys:

```env
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
SEARCH_API_KEY=<your-search-admin-key>
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_API_KEY=<your-azure-openai-key>
```

The `.env.example` file lists every variable and its default.

### Install dependencies

The alpha SDK lives on the Azure SDK public feed. The accompanying `pip.ini` and `requirements.txt` in this folder pin the exact versions used here. The cell below installs them and silences pip output for a clean notebook.

In [ ]:
%%capture
%pip install -r requirements.txt --extra-index-url https://pkgs.dev.azure.com/azure-sdk/public/_packaging/azure-sdk-for-python/pypi/simple/

## Step 1 — Configure the client

Load the `.env`, resolve sensible defaults for optional values, and build a single `AzureKeyCredential` and `SearchIndexClient` that the rest of the notebook reuses.

`azure_openai_resource_uri()` trims any `/openai/...` path off the endpoint — handy if your `.env` already contains a full deployment URL.

In [ ]:
import os
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from dotenv import load_dotenv

load_dotenv(override=True)


def env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def azure_openai_resource_uri(endpoint: str) -> str:
    # Accept either the resource root or a full deployment URL.
    return endpoint.split("/openai/", 1)[0].rstrip("/")


# Service endpoints + keys
SEARCH_ENDPOINT = env("SEARCH_ENDPOINT")
SEARCH_API_KEY = env("SEARCH_API_KEY")
AOAI_ENDPOINT = azure_openai_resource_uri(env("AOAI_ENDPOINT"))
AOAI_API_KEY = env("AOAI_API_KEY")

# Model deployments (override in .env if your deployment names differ)
EMBEDDING_MODEL = os.getenv("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.getenv("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.getenv("AOAI_GPT_MODEL", "gpt-4o")
GPT_DEPLOYMENT = os.getenv("AOAI_GPT_DEPLOYMENT", "gpt-4o")

# Resource names — all three are created in this notebook and deleted at the end
INDEX_NAME = os.getenv("INDEX_NAME", "earth-at-night")
KNOWLEDGE_SOURCE_NAME = os.getenv("KNOWLEDGE_SOURCE_NAME", "earth-knowledge-source")
KNOWLEDGE_BASE_NAME = os.getenv("KNOWLEDGE_BASE_NAME", "earth-knowledge-base")

credential = AzureKeyCredential(SEARCH_API_KEY)
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

print(f"Search service : {SEARCH_ENDPOINT}")
print(f"Foundry / AOAI : {AOAI_ENDPOINT}")
print(f"Embeddings     : {EMBEDDING_DEPLOYMENT} ({EMBEDDING_MODEL})")
print(f"Chat model     : {GPT_DEPLOYMENT} ({GPT_MODEL})")

Search service : https://fsunavala-srch-demos-prod.search.windows.net
Foundry / AOAI : https://fsunavala-openai-swecen.openai.azure.com
Embeddings     : text-embedding-3-large (text-embedding-3-large)
Chat model     : gpt-4o (gpt-4o)


## Step 2 — Create the search index

A Knowledge Source needs an underlying index to search over. We declare four fields:

| Field | Purpose |
|---|---|
| `id` | Document key |
| `page_chunk` | Searchable text content |
| `page_embedding_text_3_large` | 3072-dim vector embedding |
| `page_number` | Filterable page metadata for citations |

Three configurations make this index *agentic-ready*:

- **`vector_search`** — HNSW algorithm plus an `AzureOpenAIVectorizer` so the service can embed query strings on the fly. You never have to embed at query time.
- **`semantic_search`** — required for agentic retrieval. The Knowledge Base uses the semantic ranker to rerank candidates before synthesis.
- A single `default_configuration_name` on the semantic search block so the Knowledge Source can find it without extra plumbing.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    VectorSearch,
    VectorSearchProfile,
)

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
        sortable=True,
        facetable=True,
    ),
    SearchField(name="page_chunk", type=SearchFieldDataType.String),
    SearchField(
        name="page_embedding_text_3_large",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=3072,
        vector_search_profile_name="hnsw_text_3_large",
    ),
    SimpleField(
        name="page_number",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True,
        facetable=True,
    ),
]

embedding_parameters = AzureOpenAIVectorizerParameters(
    resource_url=AOAI_ENDPOINT,
    deployment_name=EMBEDDING_DEPLOYMENT,
    api_key=AOAI_API_KEY,
    model_name=EMBEDDING_MODEL,
)

vector_search = VectorSearch(
    profiles=[
        VectorSearchProfile(
            name="hnsw_text_3_large",
            algorithm_configuration_name="alg",
            vectorizer_name="azure_openai_text_3_large",
        )
    ],
    algorithms=[HnswAlgorithmConfiguration(name="alg")],
    vectorizers=[
        AzureOpenAIVectorizer(
            vectorizer_name="azure_openai_text_3_large",
            parameters=embedding_parameters,
        )
    ],
)

semantic_search = SemanticSearch(
    default_configuration_name="semantic_config",
    configurations=[
        SemanticConfiguration(
            name="semantic_config",
            prioritized_fields=SemanticPrioritizedFields(
                content_fields=[SemanticField(field_name="page_chunk")]
            ),
        )
    ],
)

index_client.create_or_update_index(
    SearchIndex(
        name=INDEX_NAME,
        fields=fields,
        vector_search=vector_search,
        semantic_search=semantic_search,
    )
)
print(f"Index '{INDEX_NAME}' is ready.")

Index 'earth-at-night-cookbook' is ready.


## Step 3 — Load the NASA *Earth at Night* dataset

The dataset is a pre-chunked JSON file derived from the open NASA e-book. Each row is one page chunk with its embedding already attached, so we can stream them straight into the index with the buffered sender.

In [ ]:
import requests
from azure.search.documents import SearchIndexingBufferedSender

DATA_URL = (
    "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data"
    "/refs/heads/main/nasa-e-book/earth-at-night-json/documents.json"
)
documents = requests.get(DATA_URL, timeout=60).json()

with SearchIndexingBufferedSender(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential,
) as sender:
    sender.upload_documents(documents=documents)

print(f"Uploaded {len(documents)} documents to '{INDEX_NAME}'.")

Uploaded 194 documents to 'earth-at-night-cookbook'.


## Step 4 — Create the Knowledge Source

A **Knowledge Source** is a thin pointer that tells Foundry IQ *where* a piece of data lives and *which fields* to surface as citations. It does **not** store data itself.

We declare the three fields we want to flow back in every reference:

- `id` for traceability
- `page_chunk` so the LLM sees the underlying text during synthesis
- `page_number` so the answer can cite a real page

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="NASA Earth at Night e-book — chunked pages with vector embeddings.",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        semantic_configuration_name="semantic_config",
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="page_chunk"),
            SearchIndexFieldReference(name="page_number"),
        ],
    ),
)

index_client.create_or_update_knowledge_source(knowledge_source)
print(f"Knowledge source '{KNOWLEDGE_SOURCE_NAME}' is ready.")

Knowledge source 'earth-knowledge-source-cookbook' is ready.


## Step 5 — Create the Knowledge Base

A **Knowledge Base** wraps one or more Knowledge Sources with an **LLM** and a **retrieval strategy**. The choices that matter:

| Setting | What it controls |
|---|---|
| `models` | Chat model used for query planning and answer synthesis |
| `knowledge_sources` | Which sources are in scope for this KB |
| `retrieval_reasoning_effort` | How much the planner reasons (`minimal`, `low`, `medium`) |
| `output_mode` | `extractiveData` (raw chunks) or `answerSynthesis` (cited answer) |
| `answer_instructions` | System prompt that shapes the synthesized answer |

We pick **`answerSynthesis`** so the response is a natural-language answer with citations rather than a chunk dump.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
)

gpt_parameters = AzureOpenAIVectorizerParameters(
    resource_url=AOAI_ENDPOINT,
    deployment_name=GPT_DEPLOYMENT,
    api_key=AOAI_API_KEY,
    model_name=GPT_MODEL,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=gpt_parameters)],
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode="answerSynthesis",
    answer_instructions=(
        "Provide a two sentence concise and informative answer "
        "based on the retrieved documents."
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' is ready.")

Knowledge base 'earth-knowledge-base-cookbook' is ready.


## Step 6 — Ask a complex multi-part question

This is where agentic retrieval earns its keep. The query below bundles **two unrelated sub-questions** in one turn — a classic case where naive RAG falls over because a single similarity search cannot satisfy both halves.

The Knowledge Base will:

1. **Plan** — the LLM decomposes the user turn into focused subqueries
2. **Retrieve** — each subquery hits the Search Index in parallel, with the semantic ranker reordering candidates
3. **Synthesize** — the LLM writes one grounded answer that addresses every part

We set `always_query_source=True` so the planner never short-circuits the retrieval, and `include_activity=True` so we can see what it did.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)


def make_request(messages: list[dict[str, str]]) -> KnowledgeBaseRetrievalRequest:
    return KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role=m["role"],
                content=[KnowledgeBaseMessageTextContent(text=m["content"])],
            )
            for m in messages
        ],
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                include_references=True,
                include_reference_source_data=True,
                always_query_source=True,
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    )


def answer_text(result) -> str:
    return "\n\n".join(
        content.text
        for message in (result.response or [])
        for content in (message.content or [])
        if getattr(content, "text", None)
    )


retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    credential=credential,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
)

messages = [
    {
        "role": "user",
        "content": (
            "Why do suburban belts display larger December brightening than urban "
            "cores even though absolute light levels are higher downtown? "
            "Why is the Phoenix nighttime street grid so sharply visible from space, "
            "whereas large stretches of the interstate between midwestern cities "
            "remain comparatively dim?"
        ),
    },
]

result = retrieval_client.retrieve(make_request(messages))
first_answer = answer_text(result)

print("Answer\n------")
print(first_answer)

Answer
------
December brightening is proportionally larger in suburban belts because these areas add substantial holiday and decorative lighting on top of relatively low baseline levels (yards, single‑family homes, neighborhood displays), so the percentage increase is large, whereas dense urban cores already have high, steady background illumination from streets, buildings, and signage and thus show a smaller relative increase even though their absolute light levels remain higher. [ref_id:2][ref_id:4][ref_id:6][ref_id:13]

Phoenix’s nighttime street grid appears sharply visible from space because its urban form is a large, contiguous, well-lit orthogonal grid of surface streets and arterial roads, with bright commercial nodes at intersections, whereas many Midwestern intercity interstates traverse sparsely populated rural areas where only small towns and exits are lit, leaving long stretches of highway comparatively dim and much less prominent in night-light imagery. [ref_id:0][ref_id

### Inspect the planner activity and references

The `activity` array is the audit trail — it shows every planning, search, reasoning, and synthesis step the Knowledge Base executed, with token counts and latencies. The `references` array is the citation set.

In [ ]:
import json

print(f"Activity steps : {len(result.activity or [])}")
print(f"References     : {len(result.references or [])}")
print()
print("Activity trace")
print("--------------")
print(json.dumps([a.as_dict() for a in (result.activity or [])], indent=2))

Activity steps : 6
References     : 20

Activity trace
--------------
[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1282,
    "outputTokens": 66,
    "modelName": "gpt-4o",
    "elapsedMs": 1514
  },
  {
    "type": "searchIndex",
    "id": 1,
    "knowledgeSourceName": "earth-knowledge-source-cookbook",
    "queryTime": "2026-05-18T19:12:27.7058304Z",
    "count": 14,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "December brightening suburban belts vs urban cores explanation",
      "filter": null,
      "semanticConfigurationName": null,
      "sourceDataFields": [
        {
          "name": "page_chunk"
        },
        {
          "name": "id"
        },
        {
          "name": "page_number"
        }
      ],
      "searchFields": []
    }
  },
  {
    "type": "searchIndex",
    "id": 2,
    "knowledgeSourceName": "earth-knowledge-source-cookbook",
    "queryTime": "2026-05-18T19:12:28.6393627Z",
    "count": 9,
    "elapsedMs"

In [ ]:
# Show the first few references with their page numbers — these are the citations
# that ground the answer above.
for i, ref in enumerate((result.references or [])[:3]):
    source = ref.source_data or {}
    print(f"[ref_id:{ref.id}] doc_key={ref.doc_key!r}  page={source.get('page_number')}")
    chunk = source.get("page_chunk") or ""
    snippet = chunk[:240].replace("\n", " ")
    if snippet:
        print(f"  {snippet}{'...' if len(chunk) > 240 else ''}")
    print()

[ref_id:0] doc_key='earth_at_night_508_page_104_verbalized'  page=104
  <!-- PageHeader="Urban Structure" -->  ### Location of Phoenix, Arizona  The image depicts a globe highlighting the location of Phoenix, Arizona, in the southwestern United States, marked with a blue pinpoint on the map of North America. Ph...

[ref_id:1] doc_key='earth_at_night_508_page_84_verbalized'  page=84
  <!-- PageHeader="Snow and Ice" -->  ## Snow and Ice  ### Snow Cover-Great Lakes Region, United States  An Arctic air mass brought snow to communities around the Great Lakes on December 14, 2016. The lake-effect snow came on the heels of an ...

[ref_id:3] doc_key='earth_at_night_508_page_105_verbalized'  page=105
  # Urban Structure  ## March 16, 2013  ### Phoenix Metropolitan Area at Night  This figure presents a nighttime satellite view of the Phoenix metropolitan area, highlighting urban structure and transport corridors. City lights illuminate the...



## Step 7 — Continue the conversation (multi-turn)

Agentic retrieval is conversation-aware. We append the assistant's previous answer to the message history and ask a new, narrower question. The planner uses the prior turn as context when deciding what to retrieve next.

In [ ]:
messages.append({"role": "assistant", "content": first_answer})
messages.append({"role": "user", "content": "How do I find lava at night?"})

result = retrieval_client.retrieve(make_request(messages))
second_answer = answer_text(result)

print("Follow-up answer\n----------------")
print(second_answer)
print()
print(f"Activity steps : {len(result.activity or [])}")
print(f"References     : {len(result.references or [])}")

Follow-up answer
----------------
Active lava is extremely bright in visible and infrared wavelengths, so scientists “find” it at night using satellite sensors (such as thermal and low‑light/infrared imagers) that can detect the intense heat and glow of lava flows even when they’re obscured by darkness or thin clouds. [ref_id:1][ref_id:2][ref_id:6][ref_id:9][ref_id:12] 

From the ground, the same principle applies: with clear weather and line of sight, lava fields and erupting vents stand out as distinct glowing sources against the dark surroundings, allowing them to be monitored for both scientific research and public safety during nighttime hours. [ref_id:2][ref_id:3][ref_id:6]

Activity steps : 6
References     : 24


## Step 8 — Clean up

The Knowledge Base, Knowledge Source, and Search Index are all chargeable resources. The cell below deletes them in dependency order (KB → KS → Index) so a re-run of this notebook starts from a blank slate.

Comment this cell out if you want to keep the resources around to wire up the Foundry Agent Service (see *Next steps* below).

In [ ]:
from azure.core.exceptions import ResourceNotFoundError

for name, delete_fn in [
    (KNOWLEDGE_BASE_NAME, index_client.delete_knowledge_base),
    (KNOWLEDGE_SOURCE_NAME, index_client.delete_knowledge_source),
    (INDEX_NAME, index_client.delete_index),
]:
    try:
        delete_fn(name)
        print(f"Deleted {name}.")
    except ResourceNotFoundError:
        print(f"Skipped {name} (already gone).")

Deleted earth-knowledge-base-cookbook.


Deleted earth-knowledge-source-cookbook.


Deleted earth-at-night-cookbook.


## Next steps

You now have the full **Index → Knowledge Source → Knowledge Base** pipeline working with agentic retrieval, multi-turn context, and answer synthesis. From here you can:

- **Wire the KB into the Foundry Agent Service** — the Knowledge Base exposes an MCP endpoint at `{SEARCH_ENDPOINT}/knowledgebases/{KB_NAME}/mcp?api-version=2025-11-01-Preview`. Create a `RemoteTool` project connection with `ProjectManagedIdentity` and attach it to your agent via an `MCPTool` named `knowledge_base_retrieve`. Full walkthrough: [Connect a knowledge base to a Foundry agent](https://learn.microsoft.com/azure/foundry/agents/how-to/foundry-iq-connect).
- **Add more Knowledge Sources** — the same Knowledge Base can fan out to Blob, OneLake, SharePoint (indexed or remote), and Web sources. Start with the [Knowledge Source overview](https://learn.microsoft.com/azure/search/agentic-knowledge-source-overview).
- **Tune retrieval** — raise `retrieval_reasoning_effort` to `medium` for harder queries, or switch `output_mode` to `extractiveData` if you want raw chunks for a downstream model.
- **Move to managed identity** — swap `AzureKeyCredential` for `DefaultAzureCredential` and assign *Search Service Contributor*, *Search Index Data Contributor*, and *Cognitive Services User* roles. Reference: [RBAC for Azure AI Search](https://learn.microsoft.com/azure/search/search-security-rbac).

### Reference docs

- [Agentic retrieval overview](https://learn.microsoft.com/azure/search/agentic-retrieval-overview)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Answer synthesis](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-answer-synthesis)
- [Migration: 2025-08 → 2025-11 (`knowledgeAgents` → `knowledgeBases`)](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-migrate)
